# Base-rate merged results

Explore `data/base_rate/base_rate_merged_results.csv` from a benchmark run.

Each row has **`score`** (`true`/`false`): whether the parsed answer matches **`scepticism_score_target`**. Unparseable rows have `score=false` and `parseable=false`.

In [54]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "base_rate").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MERGED_DIR = ROOT / "data" / "base_rate"
MERGED_CSV = None
for name in (
    # "base_rate_merged_results.csv",
    "base_rate_merged_results (4).csv",
):
    candidate = MERGED_DIR / name
    if candidate.is_file():
        MERGED_CSV = candidate
        break
if MERGED_CSV is None:
    raise FileNotFoundError(
        f"Missing merged results under {MERGED_DIR}. Run the base-rate benchmark first "
        "(benchmark/base-rate-benchmark.ipynb)."
    )

df = pd.read_csv(MERGED_CSV)

if "score" in df.columns:
    df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
elif "score_outcome" in df.columns:
    df["score_value"] = (df["score_outcome"] == "normative").astype(int)
else:
    raise KeyError("Merged CSV must include 'score' or legacy 'score_outcome'.")

if "parseable" in df.columns:
    df["parseable_bool"] = df["parseable"].astype(str).str.lower().eq("true")

print("Loaded:", MERGED_CSV)
print("Rows:", len(df))
print("Models:", sorted(df["model"].unique()))
print("Vignettes:", df["vignette_name"].nunique())
df.head()

Loaded: c:\src2\sceptical-llms\data\base_rate\base_rate_merged_results (4).csv
Rows: 120
Models: ['anthropic/claude-opus-4-8@default']
Vignettes: 10


,example_id,vignette_name,problem_type,intersection_size,response_type,has_statistics,variant,prompt,well_posed,normative,...,confidence_line,parsed_answer_type,parsed_percent,parsed_choice,parsed_confidence,scoring_type,parseable,score,score_value,parseable_bool
0,actor_waiter_overlap__overlap__implausible__mc_full_no_probs,actor waiter overlap,overlap,small,mc_full,False,mc_full_no_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US labor force...,False,implausible,...,4,mc_choice,NaN,A,4.0,mc_full,True,False,0,True
1,actor_waiter_overlap__overlap__implausible__mc_full_probs,actor waiter overlap,overlap,small,mc_full,True,mc_full_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US labor force...,False,implausible,...,5,mc_choice,NaN,A,5.0,mc_full,True,False,0,True
2,actor_waiter_overlap__overlap__implausible__mc_numeric_no_probs,actor waiter overlap,overlap,small,mc_numeric,False,mc_numeric_no_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US labor force...,False,implausible,...,5,mc_choice,NaN,A,5.0,mc_numeric,True,False,0,True
3,actor_waiter_overlap__overlap__implausible__mc_numeric_probs,actor waiter overlap,overlap,small,mc_numeric,True,mc_numeric_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US labor force...,False,implausible,...,5,mc_choice,NaN,A,5.0,mc_numeric,True,False,0,True
4,actor_waiter_overlap__overlap__implausible__open_no_probs,actor waiter overlap,overlap,small,open,False,open_no_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US labor force...,False,implausible,...,2,probability,0.05,NaN,2.0,open,True,False,0,True


In [55]:
df.columns

Index(['example_id', 'vignette_name', 'problem_type', 'intersection_size',
       'response_type', 'has_statistics', 'variant', 'prompt', 'well_posed',
       'normative', 'p_c_and_d_given_a', 'normative_choice',
       'normative_percent', 'normative_open', 'confidence_required',
       'numeric_score_percent', 'numeric_score_choice', 'scepticism_required',
       'scepticism_score_target', 'option_a_label', 'option_b_label',
       'option_c_label', 'option_d_label', 'option_e_label', 'option_a_lure',
       'option_b_lure', 'option_c_lure', 'option_d_lure', 'option_e_lure',
       'option_f_label', 'option_g_label', 'option_h_label', 'option_f_lure',
       'option_g_lure', 'option_h_lure', 'model', 'llm_response', 'reasoning',
       'answer_line', 'confidence_line', 'parsed_answer_type',
       'parsed_percent', 'parsed_choice', 'parsed_confidence', 'scoring_type',
       'parseable', 'score', 'score_value', 'parseable_bool'],
      dtype='str')

In [56]:
df['problem_type'].value_counts()

problem_type
overlap       60
well_posed    60
Name: count, dtype: int64

In [57]:
  df['variant'].value_counts()

variant
mc_full_no_probs       20
mc_full_probs          20
mc_numeric_no_probs    20
mc_numeric_probs       20
open_no_probs          20
open_probs             20
Name: count, dtype: int64

In [58]:
 df['intersection_size'].value_counts()

intersection_size
0         60
small     24
large     24
medium    12
Name: count, dtype: int64

## `mc_numeric_probs` detail

For each vignette: MC options A–E, the model's letter (`parsed_choice`), partition shortcut letter (`numeric_score_choice`), scepticism fields, and score.

In [59]:
MC_NUMERIC_CHOICE_COLS = [f"option_{letter}_label" for letter in "abcde"]


def format_mc_choices(row: pd.Series) -> str:
    parts = []
    for letter, col in zip("ABCDE", MC_NUMERIC_CHOICE_COLS):
        value = row.get(col)
        if pd.notna(value) and str(value).strip():
            parts.append(f"{letter}: {value}")
    return " | ".join(parts)


mc_numeric_probs = df[df["variant"] == "mc_numeric_probs"].copy()
mc_numeric_probs["choices_offered"] = mc_numeric_probs.apply(format_mc_choices, axis=1)

mc_numeric_probs_view = mc_numeric_probs[
    [
        "vignette_name",
        "choices_offered",
        "parsed_choice",
        "numeric_score_choice",
        "scepticism_required",
        "scepticism_score_target",
        "score",
        "score_value",
        "normative_choice",
        "answer_line",
    ]
].sort_values("vignette_name")

pd.set_option("display.max_colwidth", 140)
mc_numeric_probs_view

,vignette_name,choices_offered,parsed_choice,numeric_score_choice,scepticism_required,scepticism_score_target,score,score_value,normative_choice,answer_line
15,CA Trump voter,A: About 75% | B: About 63% | C: About 17% | D: About 55% | E: About 28%,NaN,A,True,NaN,False,0,A,Let me compute the joint probabilities.
21,CA Trump voter,A: About 10% | B: About 1% | C: About 4% | D: About 6% | E: About 13%,A,A,False,NaN,True,1,A,A
3,actor waiter overlap,A: About 0% | B: About 62%,A,A,True,NaN,False,0,A,A
9,actor waiter overlap,A: About 0% | B: About 19%,A,A,False,NaN,True,1,A,A
27,college STEM work,A: About 8% | B: About 11% | C: About 1% | D: About 6% | E: About 26%,E,B,True,NaN,False,0,A,E
33,college STEM work,A: About 18% | B: About 20% | C: About 3% | D: About 16% | E: About 6%,A,B,True,B,False,0,A,A
39,covid vaccine (blue/red),A: About 63% | B: About 4% | C: About 62% | D: About 8% | E: About 27%,A,A,True,NaN,False,0,A,A
45,covid vaccine (blue/red),A: About 20% | B: About 8% | C: About 14% | D: About 0% | E: About 27%,A,A,False,NaN,True,1,A,A
51,diabetes insulin obese,A: About 81% | B: About 85% | C: About 42% | D: About 83% | E: About 21%,A,B,True,NaN,False,0,A,I need to find P(diabetes | A1c > 9.0%).
57,diabetes insulin obese,A: About 56% | B: About 63% | C: About 5% | D: About 49% | E: About 42%,A,B,True,B,False,0,A,I need to find P(diabetes | A1c > 9.0%).


## `open_probs` detail

Re-parse each `llm_response` by stripping the trailing confidence line, extracting **all numeric candidates** (percents and 0–1 decimals scaled to %), and rescoring against current `benchmark.csv` targets (±0.5 pp for numeric targets).

In [67]:
import sys
from pathlib import Path

if "ROOT" not in globals():
    ROOT = Path.cwd()
    if not (ROOT / "data" / "base_rate").is_dir():
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from benchmarks.base_rate import (
    load_benchmark,
    matches_scepticism_target,
    parse_open_response,
    strip_trailing_confidence,
)

benchmark_items = load_benchmark()


def rescore_open_row(row: pd.Series) -> pd.Series:
    item = benchmark_items[row["example_id"]]
    parsed = parse_open_response(str(row["llm_response"]))
    body, confidence = strip_trailing_confidence(str(row["llm_response"]))
    matched = matches_scepticism_target(item, parsed)
    return pd.Series(
        {
            "response_body": body,
            "parsed_confidence": confidence,
            "parsed_numbers": list(parsed.percent_candidates),
            "parsed_percent_rescored": parsed.percent,
            "parsed_answer_type_rescored": parsed.answer_type,
            "target_current": item.scepticism_score_target,
            "score_rescored": matched,
            "score_value_rescored": int(matched),
        }
    )


open_probs = df[df["variant"] == "open_probs"].copy()
open_probs = pd.concat([open_probs, open_probs.apply(rescore_open_row, axis=1)], axis=1)

# Write rescored values back into the main frame for open_probs rows.
idx = open_probs.index
df.loc[idx, "parsed_percent"] = pd.to_numeric(open_probs["parsed_percent_rescored"], errors="coerce")
df.loc[idx, "parsed_answer_type"] = open_probs["parsed_answer_type_rescored"].astype("string")
parseable_rescored = open_probs["parsed_answer_type_rescored"].ne("unparseable")
df.loc[idx, "parseable"] = parseable_rescored.astype(bool)
df.loc[idx, "parseable_bool"] = parseable_rescored.astype(bool)
df.loc[idx, "score"] = open_probs["score_rescored"].astype(bool)
df.loc[idx, "score_value"] = open_probs["score_value_rescored"].astype(int)

open_probs_view = open_probs[
    [
        "vignette_name",
        "normative",
        "parsed_numbers",
        "parsed_percent_rescored",
        "parsed_confidence",
        "parsed_answer_type_rescored",
        "target_current",
        "scepticism_score_target",
        "score",
        "score_rescored",
        "score_value_rescored",
    ]
].sort_values(["vignette_name", "normative"])

print(
    "Rescored open_probs:",
    int(open_probs["score_value_rescored"].sum()),
    "/",
    len(open_probs),
)
pd.set_option("display.max_colwidth", 120)
open_probs_view

Rescored open_probs: 4 / 20


,vignette_name,normative,parsed_numbers,parsed_percent_rescored,parsed_confidence,parsed_confidence,parsed_answer_type_rescored,target_current,scepticism_score_target,score,score_rescored,score_value_rescored
17,CA Trump voter,implausible,"[80.0, 20.0, 60.0, 38.0, 98.0, 48.0, 27.0, 12.959999999999999]",12.9600,NaN,NaN,probability,meta,meta,False,False,0
23,CA Trump voter,well_posed,"[60.0, 38.0, 13.0, 7.8, 27.0, 2.106, 4.9399999999999995, 31.0, 1.5313999999999999]",1.5314,NaN,NaN,probability,9.918,9.918,False,False,0
5,actor waiter overlap,implausible,"[70.0, 30.0, 1.8, 65.0, 55.0, 16.0, 0.03, 99.97]",99.9700,NaN,NaN,probability,meta,meta,False,False,0
11,actor waiter overlap,underdetermined,"[0.03, 3.5, 30.0, 3.0]",3.0000,NaN,NaN,probability,0.03737,0.03737,True,True,1
29,college STEM work,implausible,[],NaN,4.0,4.0,meta_insufficient,meta,meta,True,True,1
35,college STEM work,underdetermined,[],NaN,4.0,4.0,meta_insufficient,20.22,20.22,False,False,0
41,covid vaccine (blue/red),implausible,"[27.0, 71.0, 19.17, 80.0, 15.336, 28.999999999999996, 7.829999999999999, 10.0, 0.783, 16.119, 73.0]",73.0000,NaN,NaN,probability,meta,meta,False,False,0
47,covid vaccine (blue/red),well_posed,"[27.0, 71.0, 8.0, 5.680000000000001, 28.999999999999996, 10.0, 2.9000000000000004, 8.58, 2.3165999999999998, 73.0, 1...",11.8066,NaN,NaN,probability,19.62,19.62,False,False,0
53,diabetes insulin obese,implausible,"[9.0, 11.0, 28.0, 47.0, 19.0, 20.0, 80.0]",80.0000,NaN,NaN,probability,meta,meta,False,False,0
59,diabetes insulin obese,underdetermined,"[9.0, 11.0, 89.0, 28.0, 47.0, 19.0]",19.0000,NaN,NaN,probability,62.67,62.67,False,False,0


In [68]:
df['reasoning'].value_counts(), df['confidence_required'].value_counts(), df['normative_choice'].value_counts()

(Series([], Name: count, dtype: int64),
 confidence_required
 True    120
 Name: count, dtype: int64,
 normative_choice
 A    80
 Name: count, dtype: int64)

## Scores by `response_type`

In [69]:
RESPONSE_TYPE_ORDER = ["open", "mc_numeric", "mc_full"]


def score_summary_table(group_col: str, *, order: list[str] | None = None) -> pd.DataFrame:
    """Counts, parseability mix, and mean score for each group value."""
    work = df.copy()
    if "parseable_bool" not in work.columns:
        work["parseable_bool"] = True
    work["score_miss"] = work["parseable_bool"] & (work["score_value"] == 0)
    work["unparseable_row"] = ~work["parseable_bool"]

    grouped = work.groupby(group_col, observed=True)
    summary = pd.DataFrame(
        {
            "n": grouped.size(),
            "score_true": grouped["score_value"].sum(),
            "score_false": grouped["score_miss"].sum(),
            "unparseable": grouped["unparseable_row"].sum(),
            "score_rate": grouped["score_value"].mean(),
        }
    )
    summary["score_pct"] = (summary["score_rate"] * 100).round(1)

    if order:
        summary = summary.reindex([value for value in order if value in summary.index])

    return summary


by_response_type = score_summary_table("response_type", order=RESPONSE_TYPE_ORDER)
by_response_type

,n,score_true,score_false,unparseable,score_rate,score_pct
response_type,,,,,,
open,40,8,29,3,0.200,20.0
mc_numeric,40,9,23,8,0.225,22.5
mc_full,40,9,21,10,0.225,22.5


## Scores by `variant`

In [70]:
VARIANT_ORDER = [
    "open_probs",
    "mc_numeric_probs",
    "mc_full_probs",
]

by_variant = score_summary_table("variant", order=VARIANT_ORDER)
by_variant

,n,score_true,score_false,unparseable,score_rate,score_pct
variant,,,,,,
open_probs,20,4,16,0,0.20,20.0
open_no_probs,20,4,13,3,0.20,20.0
mc_numeric_probs,20,5,7,8,0.25,25.0
mc_numeric_no_probs,20,4,16,0,0.20,20.0
mc_full_probs,20,4,6,10,0.20,20.0
mc_full_no_probs,20,5,15,0,0.25,25.0


## Scores by `vignette_name`

In [71]:
by_vignette = score_summary_table(
    "vignette_name",
    order=sorted(df["vignette_name"].unique()),
)
by_vignette

,n,score_true,score_false,unparseable,score_rate,score_pct
vignette_name,,,,,,
CA Trump voter,12,3,6,3,0.250000,25.0
actor waiter overlap,12,6,6,0,0.500000,50.0
college STEM work,12,3,9,0,0.250000,25.0
covid vaccine (blue/red),12,5,6,1,0.416667,41.7
diabetes insulin obese,12,2,9,1,0.166667,16.7
discharged weapon (last year),12,3,6,3,0.250000,25.0
english teacher humanities,12,2,8,2,0.166667,16.7
healthcare employment,12,0,9,3,0.000000,0.0
military overseas (federal pool),12,0,8,4,0.000000,0.0


## Optional: split by model when multiple LLMs are present

In [72]:
if df["model"].nunique() > 1:
    display(
        df.groupby(["model", "response_type"], observed=True)["score_value"]
        .mean()
        .unstack("response_type")
        .reindex(columns=RESPONSE_TYPE_ORDER)
        .round(3)
    )
    display(
        df.groupby(["model", "variant"], observed=True)["score_value"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
else:
    print("Single model in file — see tables above.")

Single model in file — see tables above.


In [73]:
df['model'].value_counts()

model
anthropic/claude-opus-4-8@default    120
Name: count, dtype: int64